# AgentCore Multi-Agent: Agent-as-Tool

> **Time budget:** ~30-40 minutes. Live session, continues directly from notebook 4.

This notebook is the harness's answer to "multi-agent." There is no supervisor/sub-agent
framework here, no Strands, no custom orchestrator: **a harness calling another harness
as a tool** is the pattern AWS recommends for the cases this workshop needs (a primary
assistant escalating narrow, well-defined requests to a specialist). If you outgrow this —
many peer agents negotiating with each other, not one agent calling one specialist — that's
where the **A2A protocol** and **code-defined agents on AgentCore Runtime** come in. Not
covered hands-on here; see the one-line mention near the end of this notebook.


## Scenario

Notebook 4's `restaurant_concierge` harness handles day-to-day menu/hours/booking
questions, and its `booking-ops` skill already tells it to "suggest contacting a human
host" for large parties (6+), private dining, and catering. Instead of just deflecting,
we give it a **specialist to consult**: a second, narrowly-scoped harness,
`restaurant_events_specialist`, that knows about the private dining room, catering
packages, and the large-party policy.

**Design choice — no new Lambda, no new Gateway.** The specialist's reference data
(`resources/lambdas/restaurant_data/data/private_events.json`) is small and static, so instead of
giving the specialist its own tool-calling loop (which would mean handling a *nested*
return-of-control cycle inside the primary harness's own tool executor — real complexity
for very little payoff in a time-boxed workshop), we bake the data directly into the
specialist's **system prompt** at build time. One harness, zero tools, zero extra infra.
If the data ever needed to be dynamic or shared across multiple specialists, promoting it
to a Gateway-backed Lambda tool (the same pattern as notebook 4's `restaurant_data`
function) would be the natural next step — mentioned again below, not built.


In [ ]:
from dotenv import load_dotenv
import json
import os
import subprocess
import sys
import uuid

import boto3

sys.path.insert(0, "resources/src")

aws_region = "us-east-1"
load_dotenv(".env")

bedrock_agentcore_client = boto3.client("bedrock-agentcore", region_name=aws_region)

PRIMARY_HARNESS_NAME = "restaurant_concierge"
PRIMARY_PROJECT_DIR = "RestaurantConciergeDemo"
SPECIALIST_HARNESS_NAME = "restaurant_events_specialist"
SPECIALIST_PROJECT_DIR = "RestaurantEventsSpecialistDemo"

os.environ["AWS_REGION"] = aws_region
os.environ["AWS_DEFAULT_REGION"] = aws_region


def run(cmd, cwd=None, check=True):
    print(f"+ {' '.join(cmd)}" + (f"  (cwd={cwd})" if cwd else ""))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        if check:
            raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result


## Step 1 — Build the private-events reference data into a system prompt

We load `resources/lambdas/restaurant_data/data/private_events.json` (already checked into the repo
as the source of truth) and render it into the specialist's `system-prompt.md`, so the
specialist always has current facts without needing a tool call.

In [ ]:
with open("resources/lambdas/restaurant_data/data/private_events.json") as f:
    private_events = json.load(f)["private_events"]

room = private_events["private_dining_room"]
packages = "\n".join(
    f"- **{p['name']}**: ${p['per_guest_usd']}/guest — {', '.join(p['includes'])}"
    for p in private_events["catering_packages"]
)

specialist_system_prompt = f"""You are the private events and catering specialist for The Bedrock Bistro.
You handle large-party reservations (6+ guests), private dining room bookings, and
catering inquiries that the general concierge escalates to you.

Reference data (do not invent numbers beyond this):
- Private dining room: "{room['name']}", capacity {room['capacity_min']}-{room['capacity_max']} guests,
  base fee ${room['base_fee_usd']}, minimum spend ${room['minimum_spend_usd']}, requires
  {room['notice_period_days']} days' notice.
- Catering packages:
{packages}
- Policy: {private_events['large_party_policy']}
- To actually confirm a booking, tell the guest a human host will follow up via:
  {private_events['contact_for_confirmation']}

Be warm, concise, and precise. You qualify the request and share options; you do not
confirm bookings yourself.
"""

print(specialist_system_prompt)


## Step 2 — Create and deploy the specialist harness

Same `agentcore` CLI flow as notebook 4's prompt-only step: a harness is just
`model` + `system prompt` + config, no orchestration code.

In [ ]:
run([
    "agentcore", "create",
    "--project-name", SPECIALIST_PROJECT_DIR,
    "--name", SPECIALIST_HARNESS_NAME,
    "--model-provider", "Bedrock",
    "--model-id", "global.anthropic.claude-sonnet-4-6",
    "--timeout", "60",
    "--max-iterations", "4",
    "--no-harness-memory",
    "--skip-git", "--skip-python-setup", "--skip-install",
], check=False)  # check=False: re-running this cell after the project already exists is fine

specialist_prompt_path = f"{SPECIALIST_PROJECT_DIR}/app/{SPECIALIST_HARNESS_NAME}/system-prompt.md"
os.makedirs(os.path.dirname(specialist_prompt_path), exist_ok=True)
with open(specialist_prompt_path, "w") as f:
    f.write(specialist_system_prompt)

print(f"Wrote {specialist_prompt_path}")


In [ ]:
run(["agentcore", "deploy"], cwd=SPECIALIST_PROJECT_DIR)


### Resolve the specialist's harness ARN

Same trick as notebook 4's observability step: read it out of the CLI's own local state
file rather than re-parsing free-text CLI output.

In [ ]:
def get_harness_arn(project_dir, harness_name):
    state_path = f"{project_dir}/agentcore/.cli/deployed-state.json"
    with open(state_path) as f:
        state = json.load(f)
    harnesses = state["targets"]["default"]["resources"]["harnesses"]
    return harnesses[harness_name]["agentRuntimeArn"]


SPECIALIST_HARNESS_ARN = get_harness_arn(SPECIALIST_PROJECT_DIR, SPECIALIST_HARNESS_NAME)
print(f"SPECIALIST_HARNESS_ARN={SPECIALIST_HARNESS_ARN}")


### Test the specialist standalone, before wiring it into anything

Always prove the piece you're about to plug in actually works on its own first.

In [ ]:
standalone_session_id = str(uuid.uuid4()) + "-standalone-check"  # >= 33 chars, per harness requirement

run([
    "agentcore", "invoke",
    "--session-id", standalone_session_id,
    "--stream",
    "We're planning a 20-person corporate dinner next month. What are our options?",
], cwd=SPECIALIST_PROJECT_DIR)


## Step 3 — Give the primary harness a way to consult the specialist

We add an **inline_function tool**, `consult_events_specialist`, to the primary
`restaurant_concierge` harness (built in notebook 4). This is the same inline-tool
mechanism notebook 4 used for `request_booking` — declarative in `harness.json`, executed
client-side.

The cell below is defensive: if you're running this notebook after notebook 4 (the normal
path), it patches the existing `harness.json`. If you're running notebook 5 standalone, it
recreates a minimal primary harness.json so the cells below still run — with a clear
warning that you're missing notebook 4's memory/booking/Gateway tools.

In [ ]:
primary_harness_path = f"{PRIMARY_PROJECT_DIR}/app/{PRIMARY_HARNESS_NAME}/harness.json"

if os.path.exists(primary_harness_path):
    with open(primary_harness_path) as f:
        primary_harness = json.load(f)
    print(f"Loaded existing {primary_harness_path} from notebook 4.")
else:
    print(
        f"WARNING: {primary_harness_path} not found — building a minimal fallback.\n"
        "Run notebook 4 first to get memory, the booking inline tool, and the Gateway-backed\n"
        "restaurant_data tool as well; this fallback only has the specialist tool."
    )
    os.makedirs(os.path.dirname(primary_harness_path), exist_ok=True)
    primary_harness = {
        "name": PRIMARY_HARNESS_NAME,
        "model": {"provider": "bedrock", "modelId": "global.anthropic.claude-sonnet-4-6"},
        "tools": [],
        "skills": [],
        "maxIterations": 10,
        "timeoutSeconds": 120,
    }

primary_harness.setdefault("tools", [])
existing = next((t for t in primary_harness["tools"] if t.get("name") == "consult_events_specialist"), None)

consult_tool = existing or {"type": "inline_function", "name": "consult_events_specialist"}
consult_tool["type"] = "inline_function"

### Your turn: describe the `consult_events_specialist` tool

Set `consult_tool["config"]` to an `inlineFunction` whose only input is `inquiry` (a string:
the guest's request, verbatim). Give it a `description` that tells the model *when* to call
it (large parties, private dining, catering) and that it should pass the guest's request
through unchanged. Then append it to `primary_harness["tools"]` if it's new, and write
`primary_harness_path`.

<details>
<summary>Click here for the solution</summary>
    
```python
consult_tool["config"] = {
    "inlineFunction": {
        "description": (
            "Consult the private events and catering specialist for large parties (6+ guests), "
            "private dining room bookings, or catering inquiries. Pass the guest's request verbatim."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "inquiry": {
                    "type": "string",
                    "description": "The guest's private-event/catering/large-party request, verbatim.",
                }
            },
            "required": ["inquiry"],
            "additionalProperties": False,
        },
    }
}
if existing is None:
    primary_harness["tools"].append(consult_tool)

with open(primary_harness_path, "w") as f:
    json.dump(primary_harness, f, indent=2)
    f.write("\n")

print(f"consult_events_specialist wired into {primary_harness_path}")
print(json.dumps(primary_harness, indent=2))
```
    
</details>

In [ ]:
run(["agentcore", "deploy"], cwd=PRIMARY_PROJECT_DIR)
PRIMARY_HARNESS_ARN = get_harness_arn(PRIMARY_PROJECT_DIR, PRIMARY_HARNESS_NAME)
print(f"PRIMARY_HARNESS_ARN={PRIMARY_HARNESS_ARN}")


## Step 4 — The client-side executor

This is the Python side of the "agent-as-tool" pattern: when the primary harness emits a
`consult_events_specialist` tool call, *your application* (this notebook) is the one that
actually calls the specialist harness — via the `bedrock-agentcore` **data-plane** boto3
client's `invoke_harness()` operation (the same API the `agentcore invoke` CLI wraps).

Write `consult_events_specialist(inquiry, actor_id="primary-harness-tool-call")`:

- build a session ID with `str(uuid.uuid4()) + "-consult-specialist"` (>= 33 chars),
- call `bedrock_agentcore_client.invoke_harness(harnessArn=SPECIALIST_HARNESS_ARN,
  runtimeSessionId=..., actorId=..., messages=[{"role": "user", "content": [{"text": inquiry}]}])`,
- `invoke_harness` returns an event stream in `response["stream"]`; accumulate every
  `event["contentBlockDelta"]["delta"]["text"]` into the final answer,
- return the accumulated answer, stripped.

<details>
<summary>Click here for the solution</summary>
    
```python
def consult_events_specialist(inquiry: str, actor_id: str = "primary-harness-tool-call") -> str:
    session_id = str(uuid.uuid4()) + "-consult-specialist"  # >= 33 chars

    response = bedrock_agentcore_client.invoke_harness(
        harnessArn=SPECIALIST_HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId=actor_id,
        messages=[{"role": "user", "content": [{"text": inquiry}]}],
    )

    answer = ""
    for event in response["stream"]:
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        text = delta.get("text")
        if text:
            answer += text
    return answer.strip()
```
    
</details>

## Step 5 — See the routing in action

**Demo A — an ordinary question.** The primary harness answers directly; it never needs
the specialist.

In [ ]:
ordinary_session_id = str(uuid.uuid4()) + "-ordinary"

run([
    "agentcore", "invoke",
    "--session-id", ordinary_session_id,
    "--stream",
    "What time do you open on Saturdays?",
], cwd=PRIMARY_PROJECT_DIR)


**Demo B — a large-party request.** Ask with `--json` so the raw tool-call event is
visible, instead of only the final streamed text. Wiring a fully automatic
return-of-control loop (harness pauses -> client executes -> client resumes the *same*
turn with the tool result appended to `messages`) is a few more lines on top of the same
`invoke_harness` call above — shown here as two explicit steps for clarity, exactly like
notebook 4's `request_booking` inline tool was demonstrated: first the tool-call request,
then the client-side handler run to show what it returns.

In [ ]:
large_party_session_id = str(uuid.uuid4()) + "-large-party"
large_party_prompt = "We'd like to book a private room for 20 people for a corporate dinner next month."

result = run([
    "agentcore", "invoke",
    "--session-id", large_party_session_id,
    "--json",
    large_party_prompt,
], cwd=PRIMARY_PROJECT_DIR, check=False)

print("---- Raw response above should show a tool-use block naming consult_events_specialist ----")


In [ ]:
# The client-side executor a full return-of-control loop would call automatically,
# run explicitly here so you can see exactly what the specialist hands back:
specialist_answer = consult_events_specialist(large_party_prompt)
print("Specialist's answer, as the primary harness would relay it to the guest:\n")
print(specialist_answer)


## Alternative wiring: Gateway-fronted agent-as-tool

The inline-function approach above works well when exactly one primary harness needs to
consult one specialist. If **multiple** different primary agents need to reach the same
specialist (e.g. a phone-support harness and a web-chat harness both escalating to the
same events specialist), front the specialist with **AgentCore Gateway** instead — the
same mechanism notebook 4 used for the `restaurant_data` Lambda — so any harness can
discover and call it as a normal Gateway tool, without every caller re-implementing its
own `invoke_harness` executor. Not built here; the inline version above is the leaner
choice for this workshop's single-primary-agent scenario.

## Next steps

Notebook 6 goes deeper on the harness's remaining platform capabilities: Skills,
Code Interpreter (replacing a hand-rolled calculator tool), a Policy guardrail, an
Evaluations run, and Observability (traces/logs) — the same `restaurant_concierge`
harness this notebook and notebook 4 built, extended rather than replaced.